# Philadelphia, PA — Land Value Tax Model

**Policy assumptions:**
- **Levy scope**: Full combined rate (city + school district) — 1.3998% = 13.998 mills applied to 100% of assessed value
- **Reform**: Split-rate 4:1 (land taxed at 4× the improvement millage rate)
- **Exemptions**: Preserve all existing exemptions — homestead (~$100K assessed-value deduction for owner-occupants), active construction abatements, and institutional/nonprofit exemptions are already embedded in OPA's `taxable_land` and `taxable_building` columns
- **Data source**: Philadelphia Office of Property Assessment (OPA) via Carto public API
- **Revenue target**: ~$796M (FY2024 current-year real estate tax collections)

**Key limitation**: OPA's land/building split defaults to 20% land for ~45% of improved parcels
(especially multi-family and commercial). This understates land value and attenuates the split-rate
impact relative to a market-derived land assessment.

In [ ]:
import sys
import io
import urllib.parse
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_current_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups
from lvt.policy_analysis import analyze_vacant_land, print_vacant_land_summary

CITY_NAME = 'philadelphia'
STATE_FIPS = '42'
COUNTY_FIPS = '101'
MODEL_TYPE = 'split_rate_4to1'
LAND_IMPROVEMENT_RATIO = 4.0
MILLAGE = 13.998  # combined city (0.6317%) + school district (0.7681%) = 1.3998%
# FY2024 city-only collections: ~$796M; school district separate; combined ~$2.1B
# We model the full combined levy — validate against modeled base directly.

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

## Section 2 — Fetch / Load Parcel Data

**Column mapping**

| Concept | Source | Column | Notes |
|---|---|---|---|
| Land value (taxable) | `assessments` year=2024 | `taxable_land` | Post-exemption; 2024 vintage matches FY2024 billing |
| Improvement value (taxable) | `assessments` year=2024 | `taxable_building` | Post-exemption; 2024 vintage |
| Total assessed value | `assessments` year=2024 | `market_value` | Pre-exemption gross |
| Use/class code | `opa_properties_public` | `category_code` | Integer 1–15 |
| Owner name(s) | `opa_properties_public` | `owner_1`, `owner_2` | Coalesced into `owner_name` in Section 3; used by the Section 6 ownership-concentration analysis |
| Geometry | `opa_properties_public` | `the_geom` | WKB point (parcel centroid) |

**Why 2024 assessments?**
The current OPA table reflects 2026 reassessments. FY2024 taxes were billed on 2024 assessments.
Using 2026 values overstates the modeled city levy by ~21% vs. actual FY2024 collections.
The Carto `assessments` history table has per-year taxable values; using `year=2024` brings
the city-levy cross-check to within ~5% of actual FY2024 collections. The residual ~5% gap
is current-year delinquency (non-collection), which is ~4–5% historically for Philadelphia.

**Assessment ratio**: 100% full market value (no assessment ratio)

**Millage source**: City of Philadelphia combined rate: 1.3998% = 13.998 mills
(city levy 0.6317% + school district 0.7681%)

In [ ]:
PARCEL_PATH = DATA_DIR / 'parcels.gpq'

if PARCEL_PATH.exists():
    gdf = gpd.read_parquet(PARCEL_PATH)
    print(f'Loaded {len(gdf):,} parcels from cache')
else:
    # Step 1: Download 2024 assessment-year taxable values (vintage used for FY2024 billing)
    print('Downloading 2024 assessment-year taxable values from Carto...')
    q_assess = (
        'SELECT parcel_number, taxable_land, taxable_building, market_value, '
        'exempt_land, exempt_building FROM assessments WHERE year = 2024'
    )
    url_assess = f'https://phl.carto.com/api/v2/sql?q={urllib.parse.quote(q_assess)}&format=csv'
    r_assess = requests.get(url_assess, timeout=300)
    r_assess.raise_for_status()
    assessments = pd.read_csv(io.StringIO(r_assess.text), low_memory=False)
    assessments['parcel_number'] = assessments['parcel_number'].astype(str)
    print(f'  {len(assessments):,} parcels in 2024 assessment year')

    # Step 2: Download current OPA for geometry, category codes, and owner name(s)
    print('Downloading OPA properties for geometry, category codes, and owner names...')
    q_opa = 'SELECT parcel_number, category_code, owner_1, owner_2, the_geom FROM opa_properties_public'
    url_opa = f'https://phl.carto.com/api/v2/sql?q={urllib.parse.quote(q_opa)}&format=csv'
    r_opa = requests.get(url_opa, timeout=300)
    r_opa.raise_for_status()
    opa_geo = pd.read_csv(io.StringIO(r_opa.text), low_memory=False)
    opa_geo['parcel_number'] = opa_geo['parcel_number'].astype(str)
    print(f'  {len(opa_geo):,} parcels in current OPA')

    # Step 3: Join assessments to OPA geometry on parcel_number
    merged = assessments.merge(opa_geo, on='parcel_number', how='inner')
    print(f'  {len(merged):,} parcels after join')

    # Step 4: Parse WKB geometry (point centroids)
    gdf = gpd.GeoDataFrame(
        merged,
        geometry=gpd.GeoSeries.from_wkb(merged['the_geom'], crs='EPSG:4326'),
        crs='EPSG:4326',
    )
    gdf = gdf.drop(columns=['the_geom'], errors='ignore')
    gdf = gdf[gdf['geometry'].notna()].copy()
    print(f'  {len(gdf):,} parcels with geometry')

    for col in ['taxable_land', 'taxable_building', 'market_value', 'exempt_land', 'exempt_building']:
        gdf[col] = pd.to_numeric(gdf[col], errors='coerce').fillna(0.0)

    gdf.to_parquet(PARCEL_PATH)
    print(f'  Cached to {PARCEL_PATH}')

print(f'Columns: {list(gdf.columns)}')
print(gdf[['taxable_land', 'taxable_building', 'market_value']].describe())

## Section 3 — Classify and Validate

In [ ]:
# Owner name: coalesce owner_1 -> owner_2 (used by the Section 6 ownership-concentration analysis)
gdf['owner_name'] = gdf[['owner_1', 'owner_2']].replace('', np.nan).bfill(axis=1).iloc[:, 0].fillna('')

# Coerce category_code to string for mapping
gdf['category_code'] = (
    pd.to_numeric(gdf['category_code'], errors='coerce')
    .astype('Int64')
    .astype(str)
)

CATEGORY_MAP = {
    '1':  'Single Family Residential',
    '2':  'Small Multi-Family (2-4 units)',
    '3':  'Mixed Use',
    '4':  'Commercial',
    '5':  'Industrial',
    '6':  'Vacant Land',
    '7':  'Other Commercial',
    '8':  'Other Residential',
    '9':  'Hotel',
    '10': 'Office / Commercial Condo',
    '11': 'Other',
    '12': 'Vacant Land',
    '13': 'Vacant Land',
    '14': 'Large Multi-Family (5+ units)',
    '15': 'Retail / General Commercial',
}
gdf['PROPERTY_CATEGORY'] = gdf['category_code'].map(CATEGORY_MAP).fillna('Other')

# Override 1: $0 improvement → Vacant Land
gdf.loc[gdf['taxable_building'] <= 0, 'PROPERTY_CATEGORY'] = 'Vacant Land'

# Override 2: separate abated improved parcels from genuinely vacant land.
GENUINE_VACANT_CODES = {'6', '12', '13'}
abated_mask = (
    (gdf['taxable_building'] <= 0) &
    (gdf['taxable_land'] > 0) &
    (~gdf['category_code'].isin(GENUINE_VACANT_CODES))
)
gdf.loc[abated_mask, 'PROPERTY_CATEGORY'] = 'Abated / Construction Exemption'

# Override 3 (NEW): OPA classifies ~1,483 parcels as vacant (codes 6/12/13) but records
# a non-zero taxable_building — small structures on lots that were never recategorized.
# Their improvement value makes them behave like improved parcels, not vacant land.
# Move them to "Improved Vacant Land" so they don't distort the Vacant Land result.
improved_vacant_mask = (
    gdf['category_code'].isin(GENUINE_VACANT_CODES) &
    (gdf['taxable_building'] > 0)
)
gdf.loc[improved_vacant_mask, 'PROPERTY_CATEGORY'] = 'Improved Vacant Land'

# Full exemption flag: both taxable values are zero
gdf['taxable_total'] = (gdf['taxable_land'] + gdf['taxable_building']).clip(lower=0)
gdf['full_exmp'] = (gdf['taxable_total'] <= 0).astype(int)

# Override 4: re-classify fully exempt parcels by their OPA type with "— Exempt" suffix.
EXEMPT_CATEGORY_MAP = {k: v + ' — Exempt' for k, v in CATEGORY_MAP.items()}
exempt_mask = gdf['full_exmp'] == 1
gdf.loc[exempt_mask, 'PROPERTY_CATEGORY'] = (
    gdf.loc[exempt_mask, 'category_code']
    .map(EXEMPT_CATEGORY_MAP)
    .fillna('Other — Exempt')
)

print(f'Total parcels: {len(gdf):,}')
print(f'Fully exempt: {gdf["full_exmp"].sum():,}  |  '
      f'Abated: {abated_mask.sum():,}  |  '
      f'Improved vacant: {improved_vacant_mask.sum():,}  |  '
      f'Taxable: {(gdf["full_exmp"] == 0).sum():,}')
print()
print('Property category distribution:')
print(gdf['PROPERTY_CATEGORY'].value_counts().to_string())

## Section 4 — Current Tax Model

Philadelphia's combined real estate tax rate is **1.3998% = 13.998 mills** applied to 100% of assessed value.

The 2024 assessment-year `taxable_land` and `taxable_building` values embed all existing exemptions
as of the 2024 billing cycle:
- **Homestead Exemption** (~$100K assessed-value deduction): reduces `taxable_building`
- **10-year construction abatement**: zeroes `taxable_building` for abated parcels
- **Institutional / nonprofit**: both columns are $0 for fully exempt parcels

Current tax = `(taxable_land + taxable_building) × 13.998 / 1000`

Cross-check: city-only portion at 0.6317% should be within ~5% of FY2024 city actuals (~$796M).
Residual gap is current-year delinquency/non-collection, not a modeling error.

In [4]:
gdf['millage_rate'] = MILLAGE

current_revenue, _, gdf = calculate_current_tax(
    df=gdf,
    tax_value_col='taxable_total',
    millage_rate_col='millage_rate',
    exemption_flag_col='full_exmp',
)

city_only_rate = 6.317  # mills (0.6317%)
city_revenue = gdf['taxable_total'].mul(city_only_rate / 1000).sum()
gap_pct = (city_revenue / 795_796_126 - 1) * 100

print(f'Modeled combined levy (city + school):  ${current_revenue:,.0f}')
print(f'Implied city-only portion (0.6317%):    ${city_revenue:,.0f}')
print(f'FY2024 city-only actuals (approx):      $795,796,126')
print(f'City portion gap: {gap_pct:+.1f}%  (target: within ±5%; residual = delinquency)')
assert abs(gap_pct) < 10.0, f'City gap {gap_pct:.1f}% exceeds 10% — check assessment year or millage'

Modeled combined levy (city + school):  $1,851,168,995
Implied city-only portion (0.6317%):    $835,393,238
FY2024 city-only actuals (approx):      $795,796,126
City portion gap: +5.0%  (target: within ±5%; residual = delinquency)


## Section 5 — Split-Rate Model (4:1)

In [5]:
# For abated parcels, impute building value using OPA's default land ratio (LR=0.20):
#   total_value = taxable_land / 0.20 = 5 × taxable_land
#   imputed_building = total_value − taxable_land = 4 × taxable_land
# Current_tax stays as actual (land-only under abatement).
# The split-rate is applied to land + imputed_building, showing what these parcels
# would owe under LVT once treated as fully assessed improved properties.
IMPUTED_BLDG_MULTIPLIER = 4.0  # building = 4 × land at OPA's 20% default land ratio

gdf['model_land'] = gdf['taxable_land'].clip(lower=0)
gdf['model_building'] = gdf['taxable_building'].clip(lower=0)
abated = gdf['PROPERTY_CATEGORY'] == 'Abated / Construction Exemption'
gdf.loc[abated, 'model_building'] = gdf.loc[abated, 'taxable_land'] * IMPUTED_BLDG_MULTIPLIER

n_imputed = abated.sum()
imputed_bldg_total = gdf.loc[abated, 'model_building'].sum()
print(f'Abated parcels with imputed building value: {n_imputed:,}')
print(f'Total imputed building base added to reform:  ${imputed_bldg_total/1e9:.2f}B')
print()

# Exclude fully-exempt parcels from the reform
taxable = gdf[gdf['full_exmp'] == 0].copy()

land_millage, improvement_millage, new_revenue, taxable = model_split_rate_tax(
    df=taxable,
    land_value_col='model_land',
    improvement_value_col='model_building',
    current_revenue=taxable['current_tax'].sum(),
    land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)

# Recombine exempt parcels (unchanged)
exempt = gdf[gdf['full_exmp'] == 1].copy()
exempt['new_tax'] = 0.0
exempt['tax_change'] = 0.0
exempt['tax_change_pct'] = 0.0
exempt['taxable_land_value'] = 0.0
exempt['taxable_improvement_value'] = 0.0
gdf = pd.concat([taxable, exempt]).sort_index()

print(f'Land millage:        {land_millage:.4f} mills')
print(f'Improvement millage: {improvement_millage:.4f} mills')
print(f'Revenue check:       ${new_revenue:,.0f} (target: ${taxable["current_tax"].sum():,.0f})')
print()

category_summary = calculate_category_tax_summary(
    df=gdf,
    category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
)
print_category_tax_summary(category_summary, title='Philadelphia — 4:1 Split-Rate Tax Impact')

Abated parcels with imputed building value: 30,109
Total imputed building base added to reform:  $12.50B



Land millage:        29.0823 mills
Improvement millage: 7.2706 mills
Revenue check:       $1,851,168,995 (target: $1,851,168,995)


Philadelphia — 4:1 Split-Rate Tax Impact


                               Category  Count Total Tax Δ ($) Total Δ (%) Mean Δ ($) Median Δ ($) Avg % Δ Median % Δ % Parcels > +10% % Parcels < -10%
              Single Family Residential 408141   $-111,669,067      -10.8%      $-274        $-183   -4.0%     -16.9%            17.3%            54.0%
         Small Multi-Family (2-4 units)  37944    $-24,230,951      -11.1%      $-639        $-520  -12.2%     -16.9%             3.8%            77.9%
        Abated / Construction Exemption  30109    $138,002,485      315.5%     $4,583       $1,396  315.5%     315.5%           100.0%             0.0%
                            Vacant Land  27927     $36,698,088      107.8%     $1,314         $422  107.8%     107.8%           100.0%             0.0%
     Single Family Residential — Exempt  27059              $0        0.0%         $0           $0    0.0%       0.0%             0.0%             0.0%
                              Mixed Use  13399     $-7,804,550      -15.6%      $-582   

## Validation Summary

| Check | Result |
|---|---|
| Revenue match | City-only cross-check within ~5% of FY2024 actuals using 2024 assessments (see Section 4) |
| Assessment vintage | 2024 (matches FY2024 billing); current OPA has 2026 values (+16% higher) |
| Residual gap | ~5% — current-year delinquency/non-collection (typical for Philadelphia) |
| Exemption handling | Homestead, abatement, institutional exemptions embedded in 2024 OPA taxable values |
| Land/improvement split | OPA official values; ~45% of improved parcels default to 20% land fraction |
| Millage rate | 13.998 mills (city 0.6317% + school 0.7681% = 1.3998%) |

## Section 6 — Ownership Concentration (Vacant Land)

Groups vacant-land parcels (`PROPERTY_CATEGORY == 'Vacant Land'`) by owner name and reports what
share of vacant land value is held by the top 5%/10% of owners, via `analyze_vacant_land()` from
`lvt.policy_analysis` — the same function already run for Baltimore. Fully exempt parcels are
excluded (`exemption_flag_col='full_exmp'`); "Improved Vacant Land" and the "— Exempt" vacant
bucket are intentionally out of scope, matching the Baltimore call's scoping.

**Limitation**: this measures concentration among distinct OPA `owner_1`/`owner_2` name strings
only. It does not resolve LLC/shell-company networks that share a beneficial owner under different
registered names, and it does not use mailing-address grouping — so any concentration figure below
is a **lower bound** on true ownership concentration.

In [ ]:
vacant_results = analyze_vacant_land(
    df=gdf,
    land_value_col='taxable_land_value',
    improvement_value_col='taxable_improvement_value',
    property_type_col='PROPERTY_CATEGORY',
    vacant_identifier='Vacant Land',
    owner_col='owner_name',
    exemption_flag_col='full_exmp',
)
print_vacant_land_summary(vacant_results)

if 'top_10_owners_by_value' in vacant_results:
    print('\nTop 10 vacant-land owners by adjusted land value:')
    print(vacant_results['top_10_owners_by_value'].to_string())

In [6]:
# Census join — must happen before export
import concurrent.futures
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

_fips = STATE_FIPS + COUNTY_FIPS
try:
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
        _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
        try:
            census_data, census_gdf = _future.result(timeout=90)
            gdf = match_to_census_blockgroups(gdf, census_gdf)
            if 'minority_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'white_pop' in gdf.columns:
                gdf['minority_pct'] = ((gdf['total_pop'] - gdf['white_pop']) / gdf['total_pop'] * 100).round(2)
            if 'black_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'black_pop' in gdf.columns:
                gdf['black_pct'] = (gdf['black_pop'] / gdf['total_pop'] * 100).round(2)
            print(f'Census join: {gdf["std_geoid"].notna().mean()*100:.1f}% matched')
        except concurrent.futures.TimeoutError:
            print('Census API timed out — skipping census join')
            for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
                gdf[_col] = float('nan')
except Exception as e:
    print(f'Census join failed: {e}')
    for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
        gdf[_col] = float('nan')

Census join: 100.0% matched


In [7]:
# Export — gdf must have census columns by this point
from lvt.lvt_utils import save_standard_export
out_df = save_standard_export(
    df=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/data/{CITY_NAME}.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
    parcel_id_col='parcel_number',
)

# Standard report — 7 PNGs in analysis/reports/philadelphia/
from lvt.viz import create_city_report
create_city_report(out_df, CITY_NAME, show=False)
print('Done.')

  [warn] philadelphia: non-standard property categories (will be preserved): ['Abated / Construction Exemption', 'Commercial — Exempt', 'Hotel — Exempt', 'Improved Vacant Land', 'Industrial — Exempt', 'Large Multi-Family (5+ units) — Exempt', 'Mixed Use — Exempt', 'Office / Commercial Condo — Exempt', 'Other Commercial — Exempt', 'Other Residential — Exempt', 'Other — Exempt', 'Single Family Residential — Exempt', 'Small Multi-Family (2-4 units) — Exempt', 'Vacant Land — Exempt']


  ✓ philadelphia: 579,815 rows → ../../analysis/data/philadelphia.csv  [model: split_rate_4to1]


Done.
